In [1]:
import os
os.chdir('/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl')

In [2]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
from importlib import reload

import interpretable_ssl.trainers.scproto_utils
import interpretable_ssl.datasets.dataset_configs
import interpretable_ssl.datasets.dataset
import interpretable_ssl.configs.defaults
import interpretable_ssl.evaluation.de_helper
import interpretable_ssl.evaluation.cd4_marker
import interpretable_ssl.evaluation.metric_helpers.embedding_metrics
import interpretable_ssl.augmenters.graph_generator
import interpretable_ssl.augmenters.adata_augmenter
import interpretable_ssl.models.swav
import interpretable_ssl.trainers.base
import interpretable_ssl.trainers.trainer
import interpretable_ssl.trainers.adaptive_trainer
import interpretable_ssl.trainers.scproto

reload(interpretable_ssl.trainers.scproto_utils)
reload(interpretable_ssl.datasets.dataset_configs)
reload(interpretable_ssl.datasets.dataset)
reload(interpretable_ssl.configs.defaults)
reload(interpretable_ssl.evaluation.de_helper)
reload(interpretable_ssl.evaluation.cd4_marker)
reload(interpretable_ssl.evaluation.metric_helpers.embedding_metrics)
reload(interpretable_ssl.augmenters.graph_generator)
reload(interpretable_ssl.augmenters.adata_augmenter)
reload(interpretable_ssl.models.swav)
reload(interpretable_ssl.trainers.base)
reload(interpretable_ssl.trainers.trainer)
reload(interpretable_ssl.trainers.adaptive_trainer)
reload(interpretable_ssl.trainers.scproto)
from interpretable_ssl.trainers.scproto import *

from interpretable_ssl.evaluation.metric_helpers.embedding_tables import *


# t = SCProtoTrainer(debug=1, workers=0, dataset_id = '0.3pancreas', full_dataset_mode = 1, affinity_type = 'arbf')
# t.setup()

 captum (see https://github.com/pytorch/captum).
INFO:faiss.loader:Loading faiss with AVX512 support.
INFO:faiss.loader:Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.


In [5]:
# problem: the niche purity, still not high enough
# observation: improving affinity, improved niche purity, but still not good enough to be better than seacell
# so we need to define affinity, with higher niche purity

In [6]:
# write a function for weighted purity, both cell type and niche
# bring load_aff here, load sarbf, c, c2, arbf, report purities

In [5]:
from interpretable_ssl.datasets.dataset_configs import *
ad, bk, lk, n = load_ds('s28nsc')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [6]:
def weighted_purity(A, labels):
    A = A.tocsr()
    labels = np.asarray(labels)
    p = np.zeros(A.shape[0])
    for i in range(A.shape[0]):
        s, e = A.indptr[i], A.indptr[i+1]
        if s == e:
            continue
        j = A.indices[s:e]
        w = A.data[s:e]
        p[i] = w[labels[j] == labels[i]].sum() / w.sum()
    return p.mean(), np.median(p)

def load_aff(at):
    p = '/home/icb/fatemehs.hashemig/codes/interpretable-ssl/graphs/'
    f = f'affinity_s28nsc58423_ncomp50_kneighbors50_{at}.pkl'
    return pkl.load(open(f'{p}/{f}', 'rb'))

def report_p(A):
    ct_w_mean, ct_w_med = weighted_purity(A, ad.obs[lk])
    ni_w_mean, ni_w_med = weighted_purity(A, ad.obs["niches_2D"])
    
    print("ct:", ct_w_mean, ct_w_med)
    print("niche:", ni_w_mean, ni_w_med)

In [18]:
sarbf = load_aff('sarbf')
report_p(sarbf)

ct: 0.5648349291192651 0.5944659752612712
niche: 0.5234296148306015 0.5201820108182585


In [19]:
c = load_aff('c')
report_p(c)

ct: 0.7640141026011573 0.8968383763516982
niche: 0.6129556295513843 0.6613634272150568


In [20]:
c2 = load_aff('c2')
report_p(c2)

ct: 0.6960980229696697 0.8446717308882549
niche: 0.7223736172316496 0.794176553742006


In [21]:
arbf = load_aff('arbf')
report_p(arbf)

ct: 0.7657101918452737 0.8866053408405987
niche: 0.3471481551775138 0.3157218366254814


In [22]:
nk = 'niches_2D'
pd.crosstab(ad.obs[lk], ad.obs[nk])

niches_2D,Airways,Alveolar spaces,Desmoplastic stroma,Excluded,Macrophage islands,Smooth muscle structures,T cell aggregates,Tumor core,Tumor surface,Vascular stroma
celltypes,,,,,,,,,,
Alveolar cells,11,444,17,43,9,0,15,3,28,37
B cells,6,5,55,4,7,4,54,0,11,28
Basal epithelial cells,681,4,6,7,0,9,9,0,12,5
Cycling immune cells,58,42,251,41,127,20,180,29,188,132
Cytotoxic T cells,164,225,1702,303,456,162,1849,46,537,567
Dendritic cells,24,24,129,18,87,1,410,9,136,26
Fibroblasts,363,415,5856,525,1047,282,1001,494,3147,2179
Lymphatic endothelial cells,8,12,391,25,20,27,85,2,38,62
Macrophages,310,388,1705,451,2501,165,900,380,1804,786


In [26]:
def check_aff(aff):
    diag_is_zero = aff.diagonal().sum() == 0
    is_symmetric = (aff != aff.T).nnz == 0
    print(f'diagonal 0: {diag_is_zero}, is_symmetric: {is_symmetric}')

def plot_aff(ad, aff, k=None, random_state=0):
    if k is not None:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(ad.n_obs, size=min(k, ad.n_obs), replace=False)
        ad = ad[idx].copy()
        aff = aff[idx][:, idx]

    ad.obsp["connectivities"] = aff.tocsr()
    ad.uns["neighbors"] = {
        "connectivities_key": "connectivities",
        "distances_key": None,
        "params": {"method": "sarb"},
    }

    sc.tl.umap(ad)
    sc.pl.umap(ad, color=[lk, "niches_2D"], wspace=1.2)

def load_aff(at):
    p = '/home/icb/fatemehs.hashemig/codes/interpretable-ssl/graphs/'
    f = f'affinity_s28nsc58423_ncomp50_kneighbors50_{at}.pkl'
    return pkl.load(open(f'{p}/{f}', 'rb'))

In [27]:
check_aff(c2)

diagonal 0: True, is_symmetric: False


In [ ]:
# i want per cell type purity of niches

In [7]:
import numpy as np
import pandas as pd
import scipy.sparse as sp


def weighted_joint_purity(A, ct, ni, groups):
    A = A.tocsr()
    ct = np.asarray(ct)
    ni = np.asarray(ni)
    groups = np.asarray(groups)

    out = {}
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        if idx.size == 0:
            continue

        p = np.zeros(idx.size)
        for k, i in enumerate(idx):
            s, e = A.indptr[i], A.indptr[i + 1]
            if s == e:
                continue
            j = A.indices[s:e]
            w = A.data[s:e]
            m = (ct[j] == ct[i]) & (ni[j] == ni[i])
            p[k] = w[m].sum() / w.sum()

        out[g] = p.mean()

    return out


def purity_tables(ad, affinities, ct_key, niche_key="niches_2D"):
    ct_rows, ni_rows, joint_rows = [], [], []

    for name, A in affinities.items():
        ct = weighted_joint_purity(A, ad.obs[ct_key], ad.obs[ct_key], ad.obs[ct_key])
        ni = weighted_joint_purity(A, ad.obs[niche_key], ad.obs[niche_key], ad.obs[ct_key])
        jt = weighted_joint_purity(A, ad.obs[ct_key], ad.obs[niche_key], ad.obs[ct_key])

        ct_rows.append(pd.Series(ct, name=name))
        ni_rows.append(pd.Series(ni, name=name))
        joint_rows.append(pd.Series(jt, name=name))

    return (
        pd.DataFrame(ct_rows),
        pd.DataFrame(ni_rows),
        pd.DataFrame(joint_rows),
    )


In [31]:
affs = {
    "arbf": arbf,
    "sarbf": sarbf,
    "c": c,
    "c2": c2
}


In [35]:
df_ct, df_niche, df_joint = purity_tables(ad, affs, lk)


In [36]:
df_joint

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium
arbf,0.367808,0.073711,0.418766,0.053161,0.176246,0.199777,0.270877,0.275090,0.199732,0.045180,0.115969,0.357258,0.282946,0.072139,0.837235,0.302124,0.391371,0.327632
sarbf,0.274053,0.040580,0.317638,0.041577,0.176366,0.167897,0.267005,0.199330,0.204302,0.036542,0.092031,0.263223,0.208715,0.050465,0.779394,0.311190,0.386796,0.252229
c,0.546770,0.114795,0.509615,0.121020,0.352255,0.373668,0.496511,0.424345,0.442970,0.113952,0.253064,0.478132,0.428491,0.148755,0.875452,0.474455,0.613169,0.535552
c2,0.532603,0.048945,0.458452,0.081544,0.368648,0.342437,0.577471,0.338131,0.486301,0.069078,0.218208,0.472369,0.373605,0.089066,0.901865,0.488220,0.645044,0.526059


In [38]:
df_ct

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium
arbf,0.587922,0.285408,0.433611,0.254851,0.637206,0.533880,0.869247,0.594127,0.790778,0.239387,0.493790,0.638440,0.725609,0.311955,0.852745,0.571438,0.926094,0.751813
sarbf,0.398281,0.145852,0.325241,0.146104,0.441591,0.344057,0.644334,0.372784,0.554321,0.138654,0.301332,0.416003,0.456062,0.177649,0.788111,0.472559,0.742512,0.483738
c,0.726275,0.236915,0.514455,0.219518,0.619971,0.529910,0.864659,0.632925,0.776323,0.223229,0.499766,0.634914,0.698772,0.269108,0.879397,0.627187,0.928285,0.761234
c2,0.661706,0.078995,0.461520,0.113144,0.512197,0.411079,0.840505,0.420310,0.692594,0.087818,0.324189,0.536735,0.484654,0.128608,0.904783,0.600447,0.890840,0.637138


In [39]:
df_niche

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium
arbf,0.424272,0.249738,0.733199,0.222218,0.262764,0.323648,0.304297,0.351663,0.245949,0.204642,0.222644,0.466879,0.382479,0.242429,0.921216,0.383811,0.408537,0.386245
sarbf,0.605054,0.495582,0.841255,0.462301,0.489521,0.563926,0.480623,0.536848,0.456400,0.450985,0.448313,0.629508,0.553198,0.476572,0.948860,0.568127,0.534390,0.573778
c,0.687383,0.493610,0.935242,0.534684,0.556474,0.651613,0.570269,0.589771,0.564117,0.491326,0.506792,0.696343,0.602933,0.544320,0.981990,0.675842,0.652331,0.665267
c2,0.762854,0.674730,0.964292,0.690932,0.707176,0.768752,0.686577,0.697168,0.697850,0.683333,0.670042,0.806802,0.724763,0.695614,0.987034,0.762415,0.718536,0.767863


In [46]:
# I am looking for a better affinity, higher rate of both niche, and cell type affinity

In [47]:
# maybe knn k = 150 on context, report both niche, ct purities
# decide how many neighbors we want? we have n samples, n meatcells
# then filter out q1
# then filter out q2 based on pca similarity
# final report

In [8]:
sc.tl.pca(ad)

In [199]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def compute_context_vectors(ad, sp_key="spatial", X_key="X_pca", k=30):
    X = ad.obsm[X_key]
    sp = ad.obsm[sp_key]

    nn = NearestNeighbors(n_neighbors=k + 1).fit(sp)
    _, I = nn.kneighbors(sp)

    C = np.empty_like(X)
    for i in range(X.shape[0]):
        C[i] = X[I[i, 1:]].mean(axis=0)

    return C
C = compute_context_vectors(ad, sp_key="spatial", X_key="X_pca", k=30)

In [183]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
import scipy.sparse as sp
import faiss

def dual_quantile_knn_seq(X1, X2, y1, y2, k1=150, q1=0.3, q2=0.2, min_n=3):
    y1 = np.asarray(y1)
    y2 = np.asarray(y2)

    X = X1.astype(np.float32)
    d = X.shape[1]
    
    index = faiss.IndexFlatL2(d)   # exact, fast
    index.add(X)
    
    d1, I = index.search(X, k1 + 1)

    n = X1.shape[0]
    nbrs = []
    p1 = np.zeros(n)
    p2 = np.zeros(n)
    nnz = np.zeros(n, int)

    for i in range(n):
        cand = I[i, 1:]
        d1_i = d1[i, 1:]

        thr1 = np.quantile(d1_i, q1)
        cand1 = cand[d1_i <= thr1]

        d2_i = ((X2[cand1] - X2[i]) ** 2).sum(1)
        d2_i = (d2_i - d2_i.min()) / (d2_i.max() - d2_i.min() + 1e-12)

        thr2 = np.quantile(d2_i, q2)
        keep = cand1[d2_i <= thr2]

        if keep.size < min_n:
            keep = cand1[np.argsort(d2_i)[:min_n]]

        nbrs.append(keep)
        nnz[i] = keep.size
        p1[i] = (y1[keep] == y1[i]).mean()
        p2[i] = (y2[keep] == y2[i]).mean()

    stats = {
        "label1_mean": p1.mean(),
        "label1_median": np.median(p1),
        "label2_mean": p2.mean(),
        "label2_median": np.median(p2),
        "nnz_mean": nnz.mean(),
        "nnz_median": np.median(nnz),
        "nnz_min": nnz.min(),
        "nnz_max": nnz.max(),
    }

    return nbrs, stats, nnz


def nbrs_to_adaptive_rbf(nbrs, X, eps=1e-12, sym="avg"):
    n = X.shape[0]
    rows = np.repeat(np.arange(n), [len(a) for a in nbrs])
    cols = np.concatenate(nbrs)
    d2 = ((X[rows] - X[cols]) ** 2).sum(1)

    sigma = np.zeros(n)
    for i in range(n):
        a = nbrs[i]
        di = ((X[a] - X[i]) ** 2).sum(1)
        sigma[i] = np.sqrt(np.median(di) + eps)

    sig_i = sigma[rows] 
    sig_j = sigma[cols]
    w = np.exp(-d2 / (sig_i * sig_j))

    A = sp.csr_matrix((w, (rows, cols)), shape=(n, n))
    A.eliminate_zeros()

    # if sym == "avg":
    #     A = (A + A.T) * 0.5
    # elif sym == "max":
    #     A = A.maximum(A.T)

    return A


In [208]:
nbrs, stats, nnz = dual_quantile_knn_seq(
    X1=C,
    X2=ad.obsm["X_pca"],
    y1=ad.obs["niches_2D"],
    y2=ad.obs[lk],
    k1=300,
    q1=0.5,
    q2=0.05,
    min_n=4,
)
stats

{'label1_mean': 0.719983955789286,
 'label1_median': 0.75,
 'label2_mean': 0.7217399055174258,
 'label2_median': 0.875,
 'nnz_mean': 8.0,
 'nnz_median': 8.0,
 'nnz_min': 8,
 'nnz_max': 8}

In [209]:
A_pca = nbrs_to_adaptive_rbf(nbrs, ad.obsm['X_pca'])
A_ctx = nbrs_to_adaptive_rbf(nbrs, C)
A = A_pca.multiply(A_ctx)



In [210]:
# A.setdiag(0)
A.eliminate_zeros()
df_ct, df_niche, df_joint = purity_tables(ad, {'new_aff': A}, lk)


In [211]:
df_joint['avg'] = df_joint.mean(axis=1)
df_joint

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium,avg
new_aff,0.617763,0.074928,0.500026,0.104649,0.405178,0.392359,0.604172,0.415418,0.530377,0.099796,0.267518,0.524117,0.430955,0.116631,0.911065,0.555688,0.674866,0.597992,0.434639


In [212]:
def add_avg(df):
    df['avg'] = df.mean(axis=1)
    return df

In [213]:
add_avg(df_niche)

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium,avg
new_aff,0.821375,0.668531,0.973176,0.700375,0.723395,0.777026,0.70042,0.728011,0.716277,0.699106,0.695041,0.826206,0.731401,0.70362,0.991091,0.845638,0.73687,0.802145,0.768873


In [214]:
add_avg(df_ct)

,Alveolar cells,B cells,Basal epithelial cells,Cycling immune cells,Cytotoxic T cells,Dendritic cells,Fibroblasts,Lymphatic endothelial cells,Macrophages,Mast cells,Monocytes,Pericytes,Plasma cells,Regulatory T cells,Respiratory epithelium,Smooth muscle cells,Tumor cells,Vascular endothelium,avg
new_aff,0.718499,0.112774,0.503262,0.138343,0.55169,0.46971,0.863588,0.51555,0.736279,0.129738,0.385411,0.593092,0.566559,0.162669,0.912908,0.601956,0.908831,0.708089,0.532164


In [216]:
pkl.dump(A, open('c3.pkl', 'wb'))

In [ ]:
# what do I need?
# for a cell, in same cell type, but also same niche

# so I will generate knn on context / pca
# then filter on same context
# then weight 

In [ ]:
# pca + knn
aff i, j = softmax(pca_dist / sigma i * sigma j)